In [ ]:
import csv
import re
import sqlite3

conn = sqlite3.connect("fx.db")

skipped = {"count":0}
neg_zero = {"count":0}
dupes = {"count":0} 

def read_rates(path):
    with open(path, newline="")as file:
        for row in csv.DictReader(file):
            yield row

def parse_currencies(rows, skipped):
    for row in rows:
        if not row["currency"] or not row["currency"].strip():
            skipped["count"] += 1
        else:
            string = row["currency"].strip()
            q = re.search(r"(\d*)\s*(.+)", string)
            quantity = 1 if q.group(1)=="" else int(q.group(1))
            currency = q.group(2)
            row["quantity"], row["currency_name"] = quantity, currency
            yield row

def clean_rates(rows, skipped, neg_zero):
     for row in rows:
            string = row["rate"].strip()
            # regex to eliminate non-digits
            row["rate"] = re.search(r'(^-?\d+\.?\d*$)', string)
            # remove none values
            if row["rate"] is None:
                skipped["count"] += 1
            else:
                row["rate"] = float(row["rate"].group(0))
                # removing negative and rate values with zero value.
                if row["rate"] <= 0:
                    neg_zero["count"] += 1
                else:
                    row["rate_per_unit"] = row["rate"]/row["quantity"]
                    yield row

def drop_duplicates(rows,dupes):
    counts = {}
    for row in rows:     
        key = (row["month"], row["currency_name"])
        counts[key] = counts.get(key,0) + 1
        if counts[key] > 1:
            dupes["count"] += 1
        else:
            yield row


def write_csv(rows, path):
        fieldnames = ["month","currency_name","rate","quantity","rate_per_unit"]
        with open(path, mode = "w", newline = "") as f:
            writer = csv.DictWriter(f, fieldnames = fieldnames, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
            

path = "fx_rates_long.csv" 
rows = drop_duplicates(clean_rates(parse_currencies(read_rates(path), skipped), skipped, neg_zero),dupes)
new_csv = write_csv(rows, "fx_rates_new.csv")



conn.execute(
    """
    CREATE TABLE IF NOT EXISTS fx_rates(
    month TEXT NOT NULL,
    currency_name TEXT NOT NULL,
    rate REAL, 
    quantity INTEGER,
    rate_per_unit REAL,
    PRIMARY KEY(month, currency_name)
    )
    """
)
rows = drop_duplicates(clean_rates(parse_currencies(read_rates(path), skipped), skipped, neg_zero),dupes)
for row in rows:
    conn.execute( 
    "INSERT OR REPLACE INTO fx_rates(month,currency_name,rate,quantity,rate_per_unit) VALUES (?,?,?,?,?)",
    (row["month"], row["currency_name"], row["rate"], row["quantity"], row["rate_per_unit"])
    )
conn.commit()
t = conn.execute(
    "SELECT COUNT(*) FROM fx_rates;"
).fetchone()
conn.close()

skipped = {"count":0}
neg_zero = {"count":0}
dupes = {"count":0} 
row_count = 0
rows = drop_duplicates(clean_rates(parse_currencies(read_rates(path), skipped), skipped, neg_zero),dupes)
for row in rows:
    row_count += 1
    
print(f"The number of rows in the database: {t[0]}")    
print(f"The number of rows retained: {row_count}")  
print(f"The number of rows skipped: {skipped["count"]}")
print(f"The number of rows skipped with negative or zero values: {neg_zero["count"]}")
print(f"The number of rows that are duplicates: {dupes["count"]}")